# Task 01 – Tensor Fundamentals
### CCS4354: Tensors and Graphs — Coursework 2026
**Prepared by:** Ahmed Aadhil
**Group Task:** Task 01 – Tensor Fundamentals (10 Marks)



## 0. Setup and Environment Check

In [ ]:
import torch
import numpy as np
import time

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device:", torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

PyTorch version: 2.11.0+cpu
CUDA available: False
Using device: cpu


## 1. Tensor Creation

A **tensor** is a multi-dimensional array — the fundamental data structure in PyTorch,
analogous to NumPy arrays but with GPU support and automatic differentiation. Below we
create tensors in several different ways, since each method is useful in a different context.

In [ ]:
# 1.1 From a Python list
t_list = torch.tensor([[1, 2, 3], [4, 5, 6]])
print("From list:\n", t_list, "\nshape:", t_list.shape, "dtype:", t_list.dtype)

From list:
 tensor([[1, 2, 3],
        [4, 5, 6]]) 
shape: torch.Size([2, 3]) dtype: torch.int64


In [ ]:
# 1.2 From a NumPy array
np_array = np.array([[1.0, 2.0], [3.0, 4.0]])
t_from_np = torch.from_numpy(np_array)
print("From NumPy:\n", t_from_np, "\ndtype:", t_from_np.dtype)

From NumPy:
 tensor([[1., 2.],
        [3., 4.]], dtype=torch.float64) 
dtype: torch.float64


In [ ]:
# 1.3 Using built-in factory functions
zeros_t = torch.zeros((2, 3))
ones_t = torch.ones((2, 3))
rand_t = torch.rand((2, 3))          # uniform [0,1)
randn_t = torch.randn((2, 3))        # standard normal
eye_t = torch.eye(3)                 # identity matrix
arange_t = torch.arange(0, 10, 2)    # like Python range
linspace_t = torch.linspace(0, 1, steps=5)

print("zeros:\n", zeros_t)
print("ones:\n", ones_t)
print("rand:\n", rand_t)
print("randn:\n", randn_t)
print("eye:\n", eye_t)
print("arange:", arange_t)
print("linspace:", linspace_t)

zeros:
 tensor([[0., 0., 0.],
        [0., 0., 0.]])
ones:
 tensor([[1., 1., 1.],
        [1., 1., 1.]])
rand:
 tensor([[0.0782, 0.0128, 0.1362],
        [0.0417, 0.3787, 0.6348]])
randn:
 tensor([[ 0.0770, -0.0481,  0.4776],
        [-1.8868, -0.8546,  1.0687]])
eye:
 tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])
arange: tensor([0, 2, 4, 6, 8])
linspace: tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])


In [ ]:
# 1.4 Specifying dtype and device explicitly
t_float32 = torch.tensor([1, 2, 3], dtype=torch.float32, device=device)
t_int64 = torch.tensor([1, 2, 3], dtype=torch.int64)
print(t_float32, t_float32.dtype, t_float32.device)
print(t_int64, t_int64.dtype)

tensor([1., 2., 3.]) torch.float32 cpu
tensor([1, 2, 3]) torch.int64


## 2. Tensor Indexing

Indexing lets us access or extract specific elements, rows, columns, or sub-tensors —
exactly how we would, for example, slice out a single node's feature vector from the
OGBN-Arxiv node feature matrix (shape `[169343, 128]`).

In [ ]:
x = torch.arange(1, 25).reshape(4, 6)
print("x:\n", x)

# Basic element indexing
print("\nElement at row 1, col 2:", x[1, 2].item())

# Row / column slicing
print("Row 0:", x[0])
print("Column 2:", x[:, 2])

# Slicing ranges
print("Rows 1-2, cols 1-3:\n", x[1:3, 1:4])

x:
 tensor([[ 1,  2,  3,  4,  5,  6],
        [ 7,  8,  9, 10, 11, 12],
        [13, 14, 15, 16, 17, 18],
        [19, 20, 21, 22, 23, 24]])

Element at row 1, col 2: 9
Row 0: tensor([1, 2, 3, 4, 5, 6])
Column 2: tensor([ 3,  9, 15, 21])
Rows 1-2, cols 1-3:
 tensor([[ 8,  9, 10],
        [14, 15, 16]])


In [ ]:
# Boolean mask indexing
mask = x % 2 == 0
print("Even mask:\n", mask)
print("Even values:", x[mask])

# Fancy (integer array) indexing
row_idx = torch.tensor([0, 2])
col_idx = torch.tensor([1, 4])
print("Fancy-indexed rows [0,2]:\n", x[row_idx])
print("Paired (row,col) picks:", x[row_idx, col_idx])

Even mask:
 tensor([[False,  True, False,  True, False,  True],
        [False,  True, False,  True, False,  True],
        [False,  True, False,  True, False,  True],
        [False,  True, False,  True, False,  True]])
Even values: tensor([ 2,  4,  6,  8, 10, 12, 14, 16, 18, 20, 22, 24])
Fancy-indexed rows [0,2]:
 tensor([[ 1,  2,  3,  4,  5,  6],
        [13, 14, 15, 16, 17, 18]])
Paired (row,col) picks: tensor([ 2, 17])


## 3. Tensor Reshaping

Reshaping changes a tensor's shape without changing its data — essential for preparing
batches of node features for a GNN layer, or flattening embeddings before a classifier head.

In [ ]:
x = torch.arange(12)
print("original:", x, x.shape)

x_reshaped = x.reshape(3, 4)
print("reshape(3,4):\n", x_reshaped)

x_view = x.view(4, 3)
print("view(4,3):\n", x_view)

x_flat = x_reshaped.flatten()
print("flatten:", x_flat)

original: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11]) torch.Size([12])
reshape(3,4):
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
view(4,3):
 tensor([[ 0,  1,  2],
        [ 3,  4,  5],
        [ 6,  7,  8],
        [ 9, 10, 11]])
flatten: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])


In [ ]:
# squeeze / unsqueeze — adding or removing dimensions of size 1
a = torch.zeros(1, 3, 1, 5)
print("original shape:", a.shape)
print("squeeze() shape:", a.squeeze().shape)          # removes all size-1 dims
print("squeeze(0) shape:", a.squeeze(0).shape)         # removes only dim 0

b = torch.zeros(3, 5)
print("unsqueeze(0) shape:", b.unsqueeze(0).shape)     # add batch dimension
print("unsqueeze(-1) shape:", b.unsqueeze(-1).shape)

original shape: torch.Size([1, 3, 1, 5])
squeeze() shape: torch.Size([3, 5])
squeeze(0) shape: torch.Size([3, 1, 5])
unsqueeze(0) shape: torch.Size([1, 3, 5])
unsqueeze(-1) shape: torch.Size([3, 5, 1])


In [ ]:
# permute / transpose — reorder dimensions
m = torch.rand(2, 3, 4)
print("original shape:", m.shape)
print("transpose(0,1) shape:", m.transpose(0, 1).shape)
print("permute(2,0,1) shape:", m.permute(2, 0, 1).shape)

original shape: torch.Size([2, 3, 4])
transpose(0,1) shape: torch.Size([3, 2, 4])
permute(2,0,1) shape: torch.Size([4, 2, 3])


## 4. Matrix Multiplication

Matrix multiplication is the core operation behind every linear layer in a neural network,
including the weight transformation `X @ W` in a GCN layer.

In [ ]:
A = torch.rand(3, 4)
B = torch.rand(4, 5)

# Standard matrix multiplication
C1 = torch.matmul(A, B)
C2 = A @ B  # equivalent shorthand
print("matmul shape:", C1.shape)
print("Equal to @ operator:", torch.allclose(C1, C2))

matmul shape: torch.Size([3, 5])
Equal to @ operator: True


In [ ]:
# Element-wise multiplication vs matrix multiplication (common source of bugs)
x = torch.tensor([[1., 2.], [3., 4.]])
y = torch.tensor([[5., 6.], [7., 8.]])

print("Element-wise (x * y):\n", x * y)
print("Matrix product (x @ y):\n", x @ y)

Element-wise (x * y):
 tensor([[ 5., 12.],
        [21., 32.]])
Matrix product (x @ y):
 tensor([[19., 22.],
        [43., 50.]])


In [ ]:
# Batch matrix multiplication — used when processing many graphs/subgraphs at once
batch_A = torch.rand(8, 3, 4)   # 8 matrices of shape (3,4)
batch_B = torch.rand(8, 4, 5)   # 8 matrices of shape (4,5)
batch_C = torch.bmm(batch_A, batch_B)
print("Batch matmul result shape:", batch_C.shape)  # (8, 3, 5)

# Dot product of two vectors
v1 = torch.tensor([1., 2., 3.])
v2 = torch.tensor([4., 5., 6.])
print("Dot product:", torch.dot(v1, v2).item())

Batch matmul result shape: torch.Size([8, 3, 5])
Dot product: 32.0


## 5. Tensor Broadcasting

Broadcasting lets PyTorch perform element-wise operations on tensors of **different but
compatible shapes** without explicitly copying data, by "stretching" the smaller tensor.

**Broadcasting rules:** Starting from the trailing (rightmost) dimension, two dimensions
are compatible if they are equal, or one of them is 1 (or missing).

In [ ]:
# Example 1: scalar broadcast
x = torch.tensor([1., 2., 3.])
print("x + 10 =", x + 10)   # scalar broadcasts to shape (3,)

# Example 2: vector broadcasting across a matrix
matrix = torch.ones(3, 4)
row_vec = torch.tensor([1., 2., 3., 4.])   # shape (4,)
print("matrix + row_vec:\n", matrix + row_vec)  # row_vec broadcasts to (3,4)

# Example 3: column vector broadcasting
col_vec = torch.tensor([[1.], [2.], [3.]])  # shape (3,1)
print("matrix + col_vec:\n", matrix + col_vec)  # broadcasts to (3,4)

x + 10 = tensor([11., 12., 13.])
matrix + row_vec:
 tensor([[2., 3., 4., 5.],
        [2., 3., 4., 5.],
        [2., 3., 4., 5.]])
matrix + col_vec:
 tensor([[2., 2., 2., 2.],
        [3., 3., 3., 3.],
        [4., 4., 4., 4.]])


In [ ]:
# Example 4: shape mismatch that CANNOT broadcast (demonstration of the rule)
try:
    bad = torch.ones(3, 4) + torch.ones(3, 5)
except RuntimeError as e:
    print("RuntimeError as expected:", e)

RuntimeError as expected: The size of tensor a (4) must match the size of tensor b (5) at non-singleton dimension 1


## 6. Tensor Aggregation Operations

Aggregation (reduction) operations collapse a tensor along one or more dimensions —
directly analogous to how a GNN aggregates neighbouring node features (sum/mean/max
aggregation in GraphSAGE, for instance).

In [ ]:
x = torch.tensor([[1., 2., 3.],
                   [4., 5., 6.]])

print("sum (all):", x.sum().item())
print("sum (dim=0, per-column):", x.sum(dim=0))
print("sum (dim=1, per-row):", x.sum(dim=1))

print("mean (all):", x.mean().item())
print("mean (dim=0):", x.mean(dim=0))

print("max (all):", x.max().item())
max_vals, max_idx = x.max(dim=1)
print("max per row:", max_vals, "at indices:", max_idx)

print("min (all):", x.min().item())
print("std (all):", x.std().item())
print("argmax (flattened):", x.argmax().item())
print("argmax (dim=1):", x.argmax(dim=1))

sum (all): 21.0
sum (dim=0, per-column): tensor([5., 7., 9.])
sum (dim=1, per-row): tensor([ 6., 15.])
mean (all): 3.5
mean (dim=0): tensor([2.5000, 3.5000, 4.5000])
max (all): 6.0
max per row: tensor([3., 6.]) at indices: tensor([2, 2])
min (all): 1.0
std (all): 1.8708287477493286
argmax (flattened): 5
argmax (dim=1): tensor([2, 2])


## 7. GPU Tensor Operations

Moving tensors to a GPU accelerates the large matrix multiplications used throughout GNN
training. We move tensors with `.to(device)` and compare CPU vs. GPU execution time.

In [ ]:
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    a_cpu = torch.rand(4000, 4000)
    b_cpu = torch.rand(4000, 4000)

    # CPU timing
    start = time.time()
    c_cpu = a_cpu @ b_cpu
    cpu_time = time.time() - start

    # Move to GPU
    a_gpu = a_cpu.to(device)
    b_gpu = b_cpu.to(device)
    torch.cuda.synchronize()

    start = time.time()
    c_gpu = a_gpu @ b_gpu
    torch.cuda.synchronize()
    gpu_time = time.time() - start

    print(f"CPU matmul time: {cpu_time:.4f}s")
    print(f"GPU matmul time: {gpu_time:.4f}s")
    print(f"Speedup: {cpu_time / gpu_time:.1f}x")

    # Moving a tensor back to CPU (e.g. before converting to NumPy)
    c_back_to_cpu = c_gpu.cpu()
    print("Result back on CPU, device:", c_back_to_cpu.device)
else:
    print("No GPU detected in this runtime — enable one via "
          "Runtime > Change runtime type > GPU, then re-run this cell.")

GPU available: False
No GPU detected in this runtime — enable one via Runtime > Change runtime type > GPU, then re-run this cell.
